# quant-lens: GPT-2 small

Does weight quantization change what GPT-2 small represents inside? Each run quantizes the
weights at 8, 6, 4 and 3 bits and compares residual-stream features under the same pretrained
SAEs (`gpt2-small-res-jb`).

Runtime > Change runtime type > **T4 GPU**, then run all cells. The model, SAE release and
hook names come from the registry entry for `gpt2`.

In [ ]:
!git clone https://github.com/iamartyaa/quant-lens.git
%cd quant-lens
!pip install -q -e .

## RTN, all 12 layers, with the extra analyses

Round-to-nearest, per output channel. All layers 0 to 11 are measured so the per-block ablation
heatmap covers the whole model. `--ablate-layers`, `--fragility` and `--steering` run at
`--analysis-bits` (default: the lowest of `--bits`, here 3).

In [ ]:
!python -m quant_lens.experiment --model gpt2 --scheme rtn \
    --layers 0 1 2 3 4 5 6 7 8 9 10 11 --bits 8 6 4 3 \
    --save-per-feature --ablate-layers --fragility --steering \
    --out results/gpt2_rtn

## AWQ-lite and W8A8 sweeps

Same bit widths at layers 2, 6 and 10. `awq_lite` calibrates on `--calib-seqs` sequences taken
from text after the evaluation sequences. `w8a8` adds simulated int8 per-token activation
quantization (`--act-bits 8`). To ablate every block for these schemes too, swap in
`--layers 0 1 2 3 4 5 6 7 8 9 10 11 --ablate-layers`.

In [ ]:
!python -m quant_lens.experiment --model gpt2 --scheme awq_lite \
    --layers 2 6 10 --bits 8 6 4 3 --calib-seqs 32 \
    --save-per-feature --out results/gpt2_awq_lite

In [ ]:
!python -m quant_lens.experiment --model gpt2 --scheme w8a8 --act-bits 8 \
    --layers 2 6 10 --bits 8 6 4 3 \
    --save-per-feature --out results/gpt2_w8a8

## Group-wise RTN

Scales per group of 128 input rows instead of per output channel.

In [ ]:
!python -m quant_lens.experiment --model gpt2 --scheme rtn --group-size 128 \
    --layers 2 6 10 --bits 8 6 4 3 \
    --save-per-feature --out results/gpt2_rtn_g128

## Reports

In [ ]:
RUN_DIRS = ['results/gpt2_rtn', 'results/gpt2_awq_lite', 'results/gpt2_w8a8', 'results/gpt2_rtn_g128']
for d in RUN_DIRS:
    !python -m quant_lens.report {d}

In [ ]:
import os
import pandas as pd
from IPython.display import Image, Markdown, display

COLS = ['hook', 'bits', 'scheme', 'group_size', 'median_feature_corr', 'jaccard', 'topk_overlap',
        'lost_frac', 'new_frac', 'fve_ref', 'fve_q', 'weight_rel_err', 'logit_kl', 'top1_agree']
IMAGES = ['summary.png', 'ablation_heatmap.png', 'fragility_scatter.png', 'steering.png']

for d in RUN_DIRS:
    if not os.path.exists(f'{d}/summary.csv'):
        print(f'{d}: no summary.csv (run skipped or failed)')
        continue
    display(Markdown(f'## {d}'))
    df = pd.read_csv(f'{d}/summary.csv')
    display(df[[c for c in COLS if c in df.columns]].round(4))
    for name in ['ablation_table.csv', 'fragility_summary.csv', 'steering.csv']:
        if os.path.exists(f'{d}/{name}'):
            display(Markdown(f'**{name}**'))
            display(pd.read_csv(f'{d}/{name}').round(4))
    for img in IMAGES:
        if os.path.exists(f'{d}/{img}'):
            display(Image(f'{d}/{img}'))

The full write-up for each run is in `<run dir>/report.md`. To keep the results, download the
`results/` folder (for example `!zip -r results.zip results`) before the runtime is recycled.